# Phase 1
1. Load COMPAS

2. Create COMPAS Query Generator

3. Create run_query_compas()

4. Create compute_spd_compas()


In [49]:
import pandas as pd
import numpy as np

In [50]:
compas = pd.read_csv("compas-scores-two-years.csv")

print(compas.shape)
print(compas.columns.tolist())

(7214, 53)
['id', 'name', 'first', 'last', 'compas_screening_date', 'sex', 'dob', 'age', 'age_cat', 'race', 'juv_fel_count', 'decile_score', 'juv_misd_count', 'juv_other_count', 'priors_count', 'days_b_screening_arrest', 'c_jail_in', 'c_jail_out', 'c_case_number', 'c_offense_date', 'c_arrest_date', 'c_days_from_compas', 'c_charge_degree', 'c_charge_desc', 'is_recid', 'r_case_number', 'r_charge_degree', 'r_days_from_arrest', 'r_offense_date', 'r_charge_desc', 'r_jail_in', 'r_jail_out', 'violent_recid', 'is_violent_recid', 'vr_case_number', 'vr_charge_degree', 'vr_offense_date', 'vr_charge_desc', 'type_of_assessment', 'decile_score.1', 'score_text', 'screening_date', 'v_type_of_assessment', 'v_decile_score', 'v_score_text', 'v_screening_date', 'in_custody', 'out_custody', 'priors_count.1', 'start', 'end', 'event', 'two_year_recid']


In [84]:
# compas = pd.read_csv("training_repairs_v3.csv")

# # print(compas.shape)
# # print(compas.columns.tolist())
# compas.head()

In [51]:
compas_df = compas[
    [
        "age",
        "sex",
        "priors_count",
        "decile_score",
        "c_charge_degree",
        "two_year_recid"
    ]
].copy()

print(compas_df.shape)
compas_df.head()

(7214, 6)


,age,sex,priors_count,decile_score,c_charge_degree,two_year_recid
0,69,Male,0,1,F,0
1,34,Male,0,3,F,1
2,24,Male,4,4,F,1
3,23,Male,1,8,F,0
4,43,Male,2,1,F,0


In [52]:
compas_df["charge_binary"] = (
    compas_df["c_charge_degree"] == "F"
).astype(int)

compas_df["charge_binary"].value_counts()

,count
charge_binary,
1,4666
0,2548


In [53]:
print(
    compas_df["two_year_recid"]
    .value_counts()
)

two_year_recid
0    3963
1    3251
Name: count, dtype: int64


In [54]:
sample_query = {
    "age_op": ">=",
    "age_th": 30,

    "priors_op": ">=",
    "priors_th": 2,

    "score_op": "<=",
    "score_th": 6,

    "charge_val": 1
}

print(sample_query)

{'age_op': '>=', 'age_th': 30, 'priors_op': '>=', 'priors_th': 2, 'score_op': '<=', 'score_th': 6, 'charge_val': 1}


In [55]:
# Equivalent to run_query_v2
def run_query_compas(data, q):

    result = data.copy()

    if q["age_op"] == ">=":
        result = result[
            result["age"] >= q["age_th"]
        ]
    else:
        result = result[
            result["age"] <= q["age_th"]
        ]

    if q["priors_op"] == ">=":
        result = result[
            result["priors_count"] >= q["priors_th"]
        ]
    else:
        result = result[
            result["priors_count"] <= q["priors_th"]
        ]

    if q["score_op"] == ">=":
        result = result[
            result["decile_score"] >= q["score_th"]
        ]
    else:
        result = result[
            result["decile_score"] <= q["score_th"]
        ]

    result = result[
        result["charge_binary"]
        == q["charge_val"]
    ]

    return result

In [56]:
sample_query = {
    "age_op": ">=",
    "age_th": 30,

    "priors_op": ">=",
    "priors_th": 2,

    "score_op": "<=",
    "score_th": 6,

    "charge_val": 1
}

In [57]:
result = run_query_compas(
    compas_df,
    sample_query
)

print("Rows:", len(result))

Rows: 1051


In [58]:
#  Equivalent of compute_spd
def compute_spd_compas(data):

    male = data[
        data["sex"] == "Male"
    ]

    female = data[
        data["sex"] == "Female"
    ]

    if len(male) == 0 or len(female) == 0:
        return None

    p_male = male["two_year_recid"].mean()

    p_female = female["two_year_recid"].mean()

    return p_male - p_female

In [59]:
spd = compute_spd_compas(result)

print("SPD:", spd)

SPD: 0.02154510794216674


The result we are getting

SPD: 0.02154510794216674

Which is way lesser that our defined threshold SPD
Now we are creating random query with COMPAS dataset for quick fairness check

In [60]:
import random

def generate_query_compas():

    return {

        "age_op":
            random.choice(
                [">=", "<="]
            ),

        "age_th":
            random.randint(
                20,
                60
            ),

        "priors_op":
            random.choice(
                [">=", "<="]
            ),

        "priors_th":
            random.randint(
                0,
                10
            ),

        "score_op":
            random.choice(
                [">=", "<="]
            ),

        "score_th":
            random.randint(
                1,
                10
            ),

        "charge_val":
            random.choice(
                [0, 1]
            )
    }

In [61]:
#  Quick fairness check
unfair = 0
fair = 0

for _ in range(100):

    q = generate_query_compas()

    result = run_query_compas(
        compas_df,
        q
    )

    if len(result) < 100:
        continue

    spd = compute_spd_compas(
        result
    )

    if spd is None:
        continue

    if abs(spd) > 0.20:
        unfair += 1
    else:
        fair += 1

print("Fair:", fair)
print("Unfair:", unfair)

Fair: 71
Unfair: 2


# Phase 2
1. Generate 100 COMPAS Queries
2. Measure SPD
3. Find Unfair Queries

In [62]:
unfair_queries = []

for _ in range(100):

    q = generate_query_compas()

    result = run_query_compas(
        compas_df,
        q
    )

    if len(result) < 100:
        continue

    spd = compute_spd_compas(
        result
    )

    if spd is None:
        continue

    if abs(spd) > 0.20:

        unfair_queries.append(
            (q, spd, len(result))
        )

print(
    "Unfair:",
    len(unfair_queries)
)

for item in unfair_queries[:5]:
    print(item)

Unfair: 6
({'age_op': '<=', 'age_th': 25, 'priors_op': '>=', 'priors_th': 0, 'score_op': '>=', 'score_th': 5, 'charge_val': 1}, np.float64(0.21183393884347557), 878)
({'age_op': '<=', 'age_th': 22, 'priors_op': '<=', 'priors_th': 7, 'score_op': '>=', 'score_th': 7, 'charge_val': 1}, np.float64(0.22495309568480293), 301)
({'age_op': '<=', 'age_th': 23, 'priors_op': '>=', 'priors_th': 0, 'score_op': '<=', 'score_th': 6, 'charge_val': 0}, np.float64(0.3234893230349841), 204)
({'age_op': '<=', 'age_th': 27, 'priors_op': '<=', 'priors_th': 4, 'score_op': '<=', 'score_th': 3, 'charge_val': 0}, np.float64(0.25875440658049353), 231)
({'age_op': '<=', 'age_th': 38, 'priors_op': '>=', 'priors_th': 10, 'score_op': '<=', 'score_th': 7, 'charge_val': 1}, np.float64(0.26074895977808604), 110)


In [63]:
FAIRNESS_THRESHOLD = 0.20

In [64]:
# Creating Distance Function
def query_distance_compas(q1, q2):

    distance = 0

    distance += abs(
        q1["age_th"]
        - q2["age_th"]
    )

    distance += abs(
        q1["priors_th"]
        - q2["priors_th"]
    )

    distance += abs(
        q1["score_th"]
        - q2["score_th"]
    )

    distance += (
        q1["charge_val"]
        != q2["charge_val"]
    )

    return distance

In [65]:
# Generating Neighbors
from itertools import product

def generate_neighbors_compas(query):

    age_values = [
        query["age_th"] - 1,
        query["age_th"],
        query["age_th"] + 1
    ]

    priors_values = [
        query["priors_th"] - 1,
        query["priors_th"],
        query["priors_th"] + 1
    ]

    score_values = [
        query["score_th"] - 1,
        query["score_th"],
        query["score_th"] + 1
    ]

    charge_values = [0, 1]

    neighbors = []

    for age, priors, score, charge in product(
        age_values,
        priors_values,
        score_values,
        charge_values
    ):

        candidate = query.copy()

        candidate["age_th"] = max(18, age)

        candidate["priors_th"] = max(0, priors)

        candidate["score_th"] = min(
            10,
            max(1, score)
        )

        candidate["charge_val"] = charge

        neighbors.append(candidate)

    return neighbors

In [66]:
neighbors = generate_neighbors_compas(
    sample_query
)

print(len(neighbors))

54


In [67]:
def find_repair_compas(query):

    neighbors = generate_neighbors_compas(
        query
    )

    best_repair = None
    best_distance = float("inf")

    for candidate in neighbors:

        result = run_query_compas(
            compas_df,
            candidate
        )

        if len(result) < 100:
            continue

        spd = compute_spd_compas(
            result
        )

        if spd is None:
            continue

        if abs(spd) <= 0.20:

            distance = query_distance_compas(
                query,
                candidate
            )

            if distance < best_distance:

                best_distance = distance

                best_repair = {
                    "repair": candidate,
                    "repair_spd": spd,
                    "selected_rows": len(result),
                    "distance": distance
                }

    return best_repair

In [68]:
# Testing on a query
test_query = {
    'age_op': '<=',
    'age_th': 21,
    'priors_op': '<=',
    'priors_th': 3,
    'score_op': '>=',
    'score_th': 4,
    'charge_val': 1
}
repair = find_repair_compas(
    test_query
)

print(repair)

{'repair': {'age_op': '<=', 'age_th': 21, 'priors_op': '<=', 'priors_th': 2, 'score_op': '>=', 'score_th': 4, 'charge_val': 1}, 'repair_spd': np.float64(0.19270798361646413), 'selected_rows': 334, 'distance': 1}


In [69]:
compas_repairs = []

In [70]:
attempts = 0

while len(compas_repairs) < 100:

    attempts += 1

    query = generate_query_compas()

    result = run_query_compas(
        compas_df,
        query
    )

    if len(result) < 100:
        continue

    spd = compute_spd_compas(
        result
    )

    if spd is None:
        continue

    if abs(spd) <= 0.20:
        continue

    repair = find_repair_compas(
        query
    )

    if repair is None:
        continue

    compas_repairs.append({

        "age_op":
            query["age_op"],

        "age_th":
            query["age_th"],

        "priors_op":
            query["priors_op"],

        "priors_th":
            query["priors_th"],

        "score_op":
            query["score_op"],

        "score_th":
            query["score_th"],

        "charge_val":
            query["charge_val"],

        "original_spd":
            spd,

        "selected_rows":
            len(result),

        "repair_age_th":
            repair["repair"]["age_th"],

        "repair_priors_th":
            repair["repair"]["priors_th"],

        "repair_score_th":
            repair["repair"]["score_th"],

        "repair_charge_val":
            repair["repair"]["charge_val"],

        "repair_spd":
            repair["repair_spd"],

        "distance":
            repair["distance"]
    })

    if len(compas_repairs) % 10 == 0:

        print(
            "Collected:",
            len(compas_repairs)
        )

print("Done")
print("Attempts:", attempts)

Collected: 10
Collected: 20
Collected: 30
Collected: 40
Collected: 50
Collected: 60
Collected: 70
Collected: 80
Collected: 90
Collected: 100
Done
Attempts: 2565


In [71]:
compas_repairs_df = pd.DataFrame(
    compas_repairs
)

print(
    compas_repairs_df.shape
)

compas_repairs_df.head()

(100, 15)


,age_op,age_th,priors_op,priors_th,score_op,score_th,charge_val,original_spd,selected_rows,repair_age_th,repair_priors_th,repair_score_th,repair_charge_val,repair_spd,distance
0,<=,29,>=,0,<=,6,0,0.204787,683,29,0,5,0,0.186454,1
1,<=,22,<=,1,>=,1,0,0.207185,169,21,1,1,0,0.041498,1
2,<=,24,<=,10,<=,3,1,0.262681,202,24,10,4,1,0.170877,1
3,<=,22,<=,2,<=,10,0,0.227246,185,21,2,10,0,0.074074,1
4,<=,42,>=,8,>=,7,0,-0.211009,119,42,7,7,0,-0.061867,1


In [72]:
compas_repairs_df.describe()

,age_th,priors_th,score_th,charge_val,original_spd,selected_rows,repair_age_th,repair_priors_th,repair_score_th,repair_charge_val,repair_spd,distance
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,33.600000,4.840000,5.600000,0.410000,0.190817,237.810000,33.430000,4.670000,5.320000,0.530000,0.113417,1.060000
std,12.425942,3.186842,2.640018,0.494311,0.182267,150.637078,12.406625,3.114174,2.522144,0.501614,0.099491,0.238683
min,20.000000,0.000000,1.000000,0.000000,-0.341880,102.000000,19.000000,0.000000,1.000000,0.000000,-0.161215,1.000000
25%,23.750000,2.000000,3.750000,0.000000,0.207851,127.500000,23.000000,2.000000,3.000000,0.000000,0.072147,1.000000
50%,28.000000,5.000000,5.000000,0.000000,0.227259,169.000000,28.000000,5.000000,5.000000,1.000000,0.155968,1.000000
75%,45.250000,8.000000,8.000000,1.000000,0.260964,297.000000,45.000000,7.000000,7.000000,1.000000,0.180091,1.000000
max,60.000000,10.000000,10.000000,1.000000,0.497101,683.000000,60.000000,10.000000,10.000000,1.000000,0.199726,2.000000


In [73]:
compas_repairs_df[[
    "original_spd",
    "repair_spd",
    "distance"
]].describe()

,original_spd,repair_spd,distance
count,100.000000,100.000000,100.000000
mean,0.190817,0.113417,1.060000
std,0.182267,0.099491,0.238683
min,-0.341880,-0.161215,1.000000
25%,0.207851,0.072147,1.000000
50%,0.227259,0.155968,1.000000
75%,0.260964,0.180091,1.000000
max,0.497101,0.199726,2.000000


In [74]:
print(compas_repairs_df.columns.tolist())

['age_op', 'age_th', 'priors_op', 'priors_th', 'score_op', 'score_th', 'charge_val', 'original_spd', 'selected_rows', 'repair_age_th', 'repair_priors_th', 'repair_score_th', 'repair_charge_val', 'repair_spd', 'distance']


In [75]:
def compute_fairness_stats_compas(data):

    total = len(data)

    if total == 0:
        return None

    male = data[
        data["sex"] == "Male"
    ]

    female = data[
        data["sex"] == "Female"
    ]

    male_ratio = len(male) / total

    female_ratio = len(female) / total

    positive_rate = (
        data["two_year_recid"]
        .mean()
    )

    male_positive_rate = (
        male["two_year_recid"]
        .mean()
        if len(male) > 0
        else 0
    )

    female_positive_rate = (
        female["two_year_recid"]
        .mean()
        if len(female) > 0
        else 0
    )

    return {
        "male_ratio":
            male_ratio,

        "female_ratio":
            female_ratio,

        "positive_rate":
            positive_rate,

        "male_positive_rate":
            male_positive_rate,

        "female_positive_rate":
            female_positive_rate
    }

In [76]:
stats = compute_fairness_stats_compas(
    result
)

print(stats)

{'male_ratio': 0.7466666666666667, 'female_ratio': 0.25333333333333335, 'positive_rate': np.float64(0.4057142857142857), 'male_positive_rate': np.float64(0.45918367346938777), 'female_positive_rate': np.float64(0.24812030075187969)}


In [77]:
result = run_query_compas(
    compas_df,
    sample_query
)

# Phase 3
1. Use Adult-Trained Model
2. Predict Repairs On COMPAS
3. Evaluate

In [78]:
compas_repairs_v2 = []

for _, row in compas_repairs_df.iterrows():

    query = {
        "age_op": row["age_op"],
        "age_th": row["age_th"],

        "priors_op": row["priors_op"],
        "priors_th": row["priors_th"],

        "score_op": row["score_op"],
        "score_th": row["score_th"],

        "charge_val": row["charge_val"]
    }

    result = run_query_compas(
        compas_df,
        query
    )

    stats = compute_fairness_stats_compas(
        result
    )

    new_row = row.to_dict()

    new_row.update(stats)

    compas_repairs_v2.append(
        new_row
    )

compas_repairs_v2 = pd.DataFrame(
    compas_repairs_v2
)

print(compas_repairs_v2.shape)

(100, 20)


In [79]:
compas_repairs_v2.to_csv(
    "compas_repairs_v2.csv",
    index=False
)

print("Saved!")

Saved!


In [80]:
compas_repairs_v2.shape
compas_repairs_v2.head()

,age_op,age_th,priors_op,priors_th,score_op,score_th,charge_val,original_spd,selected_rows,repair_age_th,repair_priors_th,repair_score_th,repair_charge_val,repair_spd,distance,male_ratio,female_ratio,positive_rate,male_positive_rate,female_positive_rate
0,<=,29,>=,0,<=,6,0,0.204787,683,29,0,5,0,0.186454,1,0.752562,0.247438,0.402635,0.453307,0.248521
1,<=,22,<=,1,>=,1,0,0.207185,169,21,1,1,0,0.041498,1,0.721893,0.278107,0.532544,0.590164,0.382979
2,<=,24,<=,10,<=,3,1,0.262681,202,24,10,4,1,0.170877,1,0.910891,0.089109,0.405941,0.429348,0.166667
3,<=,22,<=,2,<=,10,0,0.227246,185,21,2,10,0,0.074074,1,0.724324,0.275676,0.556757,0.619403,0.392157
4,<=,42,>=,8,>=,7,0,-0.211009,119,42,7,7,0,-0.061867,1,0.915966,0.084034,0.806723,0.788991,1.000000


In [45]:
from google.colab import files

files.download(
    "compas_repairs_v2.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
adult_v2 = pd.read_csv(
    "training_repairs_v2.csv"
)

print(adult_v2.shape)
adult_v2.head()

(500, 25)


,age_th,edu_th,hours_th,gain_val,original_spd,selected_rows,selectivity,male_ratio,female_ratio,positive_rate,...,repair_age_th,repair_edu_th,repair_hours_th,repair_gain_val,repair_spd,distance,delta_age,delta_edu,delta_hours,delta_gain
0,31,8,22,0,0.257169,16449,0.505175,0.704237,0.295763,0.322938,...,30,8,22,1,0.197082,1.032258,-1,0,0,1
1,41,10,44,1,0.304114,209,0.006419,0.889952,0.110048,0.488038,...,41,10,40,1,0.188799,0.090909,0,0,-4,0
2,45,9,59,0,0.235863,287,0.008814,0.885017,0.114983,0.299652,...,45,9,61,0,0.174603,0.033898,0,0,2,0
3,41,11,25,1,0.205877,728,0.022358,0.820055,0.179945,0.581044,...,40,11,25,1,0.197889,0.024390,-1,0,0,0
4,60,10,38,0,0.263678,425,0.013052,0.632941,0.367059,0.211765,...,60,9,38,0,0.169681,0.100000,0,-1,0,0


In [81]:
adult_v2[[
    "male_ratio",
    "female_ratio",
    "positive_rate",
    "male_positive_rate",
    "female_positive_rate"
]].describe()

,male_ratio,female_ratio,positive_rate,male_positive_rate,female_positive_rate
count,500.000000,500.000000,500.000000,500.000000,500.000000
mean,0.782994,0.217006,0.525638,0.577906,0.329672
std,0.076861,0.076861,0.180877,0.178162,0.177495
min,0.393162,0.056122,0.193683,0.133333,0.044872
25%,0.740198,0.163887,0.360980,0.417162,0.171994
50%,0.791784,0.208216,0.531505,0.595080,0.309237
75%,0.836113,0.259802,0.662919,0.709991,0.483763
max,0.943878,0.606838,0.926606,0.956989,0.750000


In [82]:
compas_repairs_v2[[
    "male_ratio",
    "female_ratio",
    "positive_rate",
    "male_positive_rate",
    "female_positive_rate"
]].describe()

,male_ratio,female_ratio,positive_rate,male_positive_rate,female_positive_rate
count,100.000000,100.000000,100.000000,100.000000,100.000000
mean,0.827798,0.172202,0.565098,0.601927,0.411110
std,0.078432,0.078432,0.130328,0.119725,0.228955
min,0.648148,0.037037,0.313433,0.350877,0.074074
25%,0.746058,0.110577,0.465479,0.523123,0.285606
50%,0.846631,0.153369,0.559540,0.602422,0.357379
75%,0.889423,0.253942,0.624934,0.663934,0.470965
max,0.962963,0.351852,0.843750,0.864130,1.000000


In [85]:
adult_v2["distance"].describe()

,distance
count,500.000000
mean,0.358901
std,0.428728
min,0.016667
25%,0.056604
50%,0.105263
75%,1.000000
max,1.219780


In [86]:
compas_repairs_v2["distance"].describe()

,distance
count,100.000000
mean,1.060000
std,0.238683
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,2.000000


# Phase 4
1. Apply Guided Search
2. Evaluate Again

In [87]:
adult_summary = {
    "dataset": "Adult",
    "repairs": len(adult_v2),
    "mean_original_spd": adult_v2["original_spd"].mean(),
    "mean_repair_spd": adult_v2["repair_spd"].mean(),
    "mean_male_positive": adult_v2["male_positive_rate"].mean(),
    "mean_female_positive": adult_v2["female_positive_rate"].mean()
}

compas_summary = {
    "dataset": "COMPAS",
    "repairs": len(compas_repairs_v2),
    "mean_original_spd": compas_repairs_v2["original_spd"].mean(),
    "mean_repair_spd": compas_repairs_v2["repair_spd"].mean(),
    "mean_male_positive": compas_repairs_v2["male_positive_rate"].mean(),
    "mean_female_positive": compas_repairs_v2["female_positive_rate"].mean()
}

comparison = pd.DataFrame(
    [adult_summary, compas_summary]
)

comparison

,dataset,repairs,mean_original_spd,mean_repair_spd,mean_male_positive,mean_female_positive
0,Adult,500,0.248234,0.175066,0.577906,0.329672
1,COMPAS,100,0.190817,0.113417,0.601927,0.411110
